# SOTA-Style Thin-Vessel Segmentation Training

This notebook trains a strong retinal vessel baseline for comparison against vanilla CMUNeXt: U-Net++ with a pretrained EfficientNet encoder and vessel-friendly losses.

Expected DRIVE dataset layout after Kaggle extraction:

```text
DRIVE/
  training/
    images/
      21_training.tif
    1st_manual/
      21_manual1.gif
```

Image and mask filenames are paired by their numeric DRIVE ID.

In [ ]:
!pip install -q torch torchvision tqdm pillow pandas kaggle segmentation-models-pytorch timm

API_KEY = "KGAT_aa78fbc54f07fbd6bcc5cad39350ddbf"
KAGGLE_USERNAME = "andysuri"  # Put your Kaggle username here if you are not using Colab Secrets.
CMUNEXT_REPO_ROOT = None  # Optional: set to the cloned CMUNeXt repo root if autodiscovery fails.
OUTPUT_ROOT = "/content/outputs"

import os
import sys
from pathlib import Path

# Clean Colab assumption: this notebook plus a cloned FengheTan9/CMUNeXt repo.
def discover_path(search_root, marker, max_depth=6):
    search_root = Path(search_root).expanduser()
    if not search_root.exists():
        return None
    if (search_root / marker).exists():
        return search_root.resolve()
    root_depth = len(search_root.resolve().parts)
    for current, dirs, _ in os.walk(search_root):
        current_path = Path(current)
        if len(current_path.resolve().parts) - root_depth >= max_depth:
            dirs[:] = []
            continue
        if (current_path / marker).exists():
            return current_path.resolve()
    return None

def find_path(candidates, marker, search_roots=()):
    checked = []
    for candidate in candidates:
        if candidate is None:
            continue
        path = Path(candidate).expanduser()
        checked.append(str(path))
        if (path / marker).exists():
            return path.resolve(), checked
    for search_root in search_roots:
        if search_root is None:
            continue
        checked.append(f"{search_root}/**/{marker}")
        found = discover_path(search_root, marker)
        if found is not None:
            return found, checked
    return None, checked

cwd = Path.cwd()
content_search_roots = [Path('/content'), Path('/workspace'), Path('/content/drive/MyDrive')]
cmunext_repo, _ = find_path(
    [
        CMUNEXT_REPO_ROOT,
        Path('/content/CMUNeXt'),
        Path('/content/CMUNEXT'),
        Path('/content/CMUNEXT/CMUNeXt'),
        Path('/content/CMUNEXT/CMUNEXT/CMUNeXt'),
        cwd,
        *(parent for parent in cwd.parents if str(parent) != '/'),
    ],
    Path('network') / 'CMUNeXt.py',
    search_roots=content_search_roots,
)

if cmunext_repo is not None:
    cmunext_import_path = str(cmunext_repo)
    if cmunext_import_path not in sys.path:
        sys.path.insert(0, cmunext_import_path)

# Prefer the local string variables above; fall back to env vars or Colab Secrets if they are blank.
def get_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

kaggle_username = KAGGLE_USERNAME or os.environ.get('KAGGLE_USERNAME') or get_colab_secret('KAGGLE_USERNAME')
kaggle_key = API_KEY or os.environ.get('KAGGLE_KEY') or get_colab_secret('KAGGLE_KEY')
if not kaggle_username or not kaggle_key:
    raise RuntimeError('Set KAGGLE_USERNAME and API_KEY in this cell, or add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets.')
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

from kaggle.api.kaggle_api_extended import KaggleApi

# Kaggle dataset: https://www.kaggle.com/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction
KAGGLE_DATASET = 'andrewmvd/drive-digital-retinal-images-for-vessel-extraction'
DATASET_SUBDIR = ''

dataset_dir = Path('/content/datasets') / KAGGLE_DATASET.split('/')[-1]
dataset_dir.mkdir(parents=True, exist_ok=True)

api = KaggleApi()
api.authenticate()
if not any(dataset_dir.iterdir()):
    api.dataset_download_files(KAGGLE_DATASET, path=dataset_dir, unzip=True, quiet=False)

DATA_ROOT = dataset_dir / DATASET_SUBDIR if DATASET_SUBDIR else dataset_dir
IMAGE_DIR = DATA_ROOT / 'images'
MASK_DIR = DATA_ROOT / 'masks'

OUTPUT_DIR = Path(OUTPUT_ROOT) / 'sota_unetpp'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('CMUNeXt repo:', cmunext_repo if cmunext_repo else 'not found; not required for the SOTA baseline')
print('Dataset root:', DATA_ROOT)
print('Images:', IMAGE_DIR)
print('Masks:', MASK_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
!mkdir CMUNEXT ; cd CMUNEXT ; git clone https://github.com/FengheTan9/CMUNeXt ; cd ../

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.gif'}


def drive_id_from_filename(path):
    """
    DRIVE files usually look like:
    image: 21_training.tif
    vessel mask: 21_manual1.gif

    This extracts the shared numeric ID: 21
    """
    stem = path.stem
    return stem.split('_')[0]


def split_image_mask_pairs(image_dir, mask_dir, val_fraction=0.2, seed=41):
    image_dir = Path(image_dir)
    mask_dir = Path(mask_dir)

    if not image_dir.exists():
        raise FileNotFoundError(f'Image directory not found: {image_dir}')

    if not mask_dir.exists():
        raise FileNotFoundError(f'Mask directory not found: {mask_dir}')

    masks_by_id = {
        drive_id_from_filename(path): path
        for path in mask_dir.iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
    }

    pairs = []

    for image_path in image_dir.iterdir():
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        image_id = drive_id_from_filename(image_path)

        if image_id in masks_by_id:
            pairs.append((image_path, masks_by_id[image_id]))

    if not pairs:
        raise ValueError(
            f'No image/mask pairs found.\n'
            f'Image dir: {image_dir}\n'
            f'Mask dir: {mask_dir}\n'
            f'DRIVE image names look like 21_training.tif and masks like 21_manual1.gif.'
        )

    rng = random.Random(seed)
    rng.shuffle(pairs)

    val_count = max(1, int(round(len(pairs) * val_fraction))) if len(pairs) > 1 else 0

    train_pairs = pairs[val_count:]
    val_pairs = pairs[:val_count]

    return train_pairs, val_pairs


class VesselSegmentationDataset(Dataset):
    def __init__(self, pairs, image_size=512, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]
        image = Image.open(image_path).convert('RGB').resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = Image.open(mask_path).convert('L').resize((self.image_size, self.image_size), Image.NEAREST)

        image_array = np.asarray(image, dtype=np.float32) / 255.0
        mask_array = (np.asarray(mask, dtype=np.float32) > 127).astype(np.float32)

        if self.augment and random.random() < 0.5:
            image_array = np.flip(image_array, axis=1).copy()
            mask_array = np.flip(mask_array, axis=1).copy()
        if self.augment and random.random() < 0.5:
            image_array = np.flip(image_array, axis=0).copy()
            mask_array = np.flip(mask_array, axis=0).copy()

        image_tensor = torch.from_numpy(image_array).permute(2, 0, 1)
        mask_tensor = torch.from_numpy(mask_array).unsqueeze(0)
        return image_tensor, mask_tensor


class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).flatten(1)
        targets = targets.flatten(1)
        intersection = (probs * targets).sum(dim=1)
        denominator = probs.sum(dim=1) + targets.sum(dim=1)
        dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
        return 1.0 - dice.mean()


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.dice = SoftDiceLoss()

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        dice = self.dice(logits, targets)
        return self.bce_weight * bce + self.dice_weight * dice


class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).flatten(1)
        targets = targets.flatten(1)
        tp = (probs * targets).sum(dim=1)
        fp = (probs * (1.0 - targets)).sum(dim=1)
        fn = ((1.0 - probs) * targets).sum(dim=1)
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return 1.0 - tversky.mean()


def count_trainable_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


@torch.no_grad()
def binary_segmentation_metrics(logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    targets = (targets >= 0.5).float()

    tp = (preds * targets).sum()
    fp = (preds * (1.0 - targets)).sum()
    fn = ((1.0 - preds) * targets).sum()
    tn = ((1.0 - preds) * (1.0 - targets)).sum()

    dice = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    sensitivity = (tp + eps) / (tp + fn + eps)
    specificity = (tn + eps) / (tn + fp + eps)
    accuracy = (tp + tn + eps) / (tp + tn + fp + fn + eps)

    return {
        'dice': float(dice.item()),
        'iou': float(iou.item()),
        'sensitivity': float(sensitivity.item()),
        'specificity': float(specificity.item()),
        'accuracy': float(accuracy.item()),
    }


def build_sota_unetpp(encoder_name='efficientnet-b0', encoder_weights='imagenet', in_channels=3, classes=1):
    import segmentation_models_pytorch as smp
    return smp.UnetPlusPlus(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=classes,
        activation=None,
    )


SEED = 41
IMAGE_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 50
LR = 1e-4
VAL_FRACTION = 0.2
ENCODER_NAME = 'efficientnet-b0'

DATASETS_ROOT_CANDIDATES = [
    Path('/content/datasets'),
    Path('/content/dataset'),
    Path('/content'),
]

drive_roots = []

for root in DATASETS_ROOT_CANDIDATES:
    if root.exists():
        drive_roots.extend(root.rglob('DRIVE'))

drive_roots = [
    path for path in drive_roots
    if path.is_dir()
    and (path / 'training').exists()
    and (path / 'training' / 'images').exists()
    and (path / 'training' / '1st_manual').exists()
]

if not drive_roots:
    print('Could not auto-find DRIVE folder.')
    print()
    print('Existing folders under /content:')
    for path in Path('/content').rglob('*'):
        if path.is_dir() and path.name.lower() in {'datasets', 'dataset', 'drive', 'training', 'images', '1st_manual', 'mask'}:
            print(path)

    raise FileNotFoundError(
        'Could not find a valid DRIVE folder. '
        'Expected structure: DRIVE/training/images and DRIVE/training/1st_manual'
    )

DRIVE_ROOT = drive_roots[0]

IMAGE_DIR = DRIVE_ROOT / 'training' / 'images'
MASK_DIR = DRIVE_ROOT / 'training' / '1st_manual'

print('DRIVE_ROOT:', DRIVE_ROOT)
print('IMAGE_DIR:', IMAGE_DIR)
print('MASK_DIR:', MASK_DIR)
print('Training image count:', len(list(IMAGE_DIR.iterdir())))
print('Manual mask count:', len(list(MASK_DIR.iterdir())))

IMAGE_DIR = DRIVE_ROOT / 'training' / 'images'
MASK_DIR = DRIVE_ROOT / 'training' / '1st_manual'

print('DRIVE_ROOT:', DRIVE_ROOT)
print('IMAGE_DIR:', IMAGE_DIR)
print('MASK_DIR:', MASK_DIR)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
train_pairs, val_pairs = split_image_mask_pairs(
    IMAGE_DIR,
    MASK_DIR,
    val_fraction=VAL_FRACTION,
    seed=SEED,
)

train_ds = VesselSegmentationDataset(train_pairs, image_size=IMAGE_SIZE, augment=True)
val_ds = VesselSegmentationDataset(val_pairs, image_size=IMAGE_SIZE, augment=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print('Train images:', len(train_ds))
print('Val images:', len(val_ds))

In [ ]:
class VesselAwareLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce_dice = BCEDiceLoss(bce_weight=0.4, dice_weight=0.6)
        self.tversky = TverskyLoss(alpha=0.3, beta=0.7)

    def forward(self, logits, targets):
        return 0.6 * self.bce_dice(logits, targets) + 0.4 * self.tversky(logits, targets)


model = build_sota_unetpp(
    encoder_name=ENCODER_NAME,
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
).to(device)

criterion = VesselAwareLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print('Encoder:', ENCODER_NAME)
print('Trainable parameters:', count_trainable_parameters(model))

In [ ]:
def run_validation():
    model.eval()
    loss_total = 0.0
    metric_totals = {'dice': 0.0, 'iou': 0.0, 'sensitivity': 0.0, 'specificity': 0.0, 'accuracy': 0.0}
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            logits = model(images)
            loss_total += criterion(logits, masks).item()
            metrics = binary_segmentation_metrics(logits, masks)
            for key in metric_totals:
                metric_totals[key] += metrics[key]
    count = max(1, len(val_loader))
    return loss_total / count, {key: value / count for key, value in metric_totals.items()}

best_dice = -1.0
history = []
history_csv = OUTPUT_DIR / 'training_history.csv'

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        images = images.to(device)
        masks = masks.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    val_loss, val_metrics = run_validation()
    train_loss /= max(1, len(train_loader))
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, **val_metrics}
    history.append(row)
    pd.DataFrame(history).to_csv(history_csv, index=False)
    print(row)

    if val_metrics['dice'] > best_dice:
        best_dice = val_metrics['dice']
        torch.save(
            {'model_state_dict': model.state_dict(), 'config': {'image_size': IMAGE_SIZE, 'encoder_name': ENCODER_NAME}},
            OUTPUT_DIR / 'best_sota_unetpp.pt',
        )
        print('Saved best checkpoint')

In [ ]:
history_df = pd.read_csv(OUTPUT_DIR / 'training_history.csv')
history_df.tail()